I use the https://www.manythings.org/anki/ dataset for sentence pairs of english and russian.

In [ ]:

import os
import urllib.request
import zipfile
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import math
import time
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download('punkt', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

#fix random seeds
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

if not os.path.exists("rus-eng.zip"):
    url = "https://www.manythings.org/anki/rus-eng.zip"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as response, open("rus-eng.zip", 'wb') as out_file:
        out_file.write(response.read())
    print("Downloaded")

if not os.path.exists("rus.txt"):
    with zipfile.ZipFile("rus-eng.zip", 'r') as zip_ref:
        zip_ref.extractall(".")

lines = open('rus.txt', encoding='utf-8').read().strip().split('\n')
pairs = [[s for s in line.split('\t')[:2]] for line in lines[:10000]]

def preprocess(s):
    s = s.lower().strip()
    s = re.sub(r"([?.!,¿])", r" \1 ", s)
    s = re.sub(r'[" "]+', " ", s)
    s = re.sub(r"[^a-zA-Zа-яё?.!,¿]+", " ", s)
    return s.strip()

eng_sentences = [preprocess(p[0]) for p in pairs]
rus_sentences = [preprocess(p[1]) for p in pairs]

print(f"Total sentences:", len(eng_sentences))
for i in range(5):
    print("Example:", eng_sentences[i], " ", rus_sentences[i])

Using device: cuda
Total sentences: 10000
Example: go .   марш !
Example: go .   иди .
Example: go .   идите .
Example: hi .   здравствуйте .
Example: hi .   привет !


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    if Q.ndim == 2:
        Q = Q[np.newaxis, :, :]
        K = K[np.newaxis, :, :]
        V = V[np.newaxis, :, :]

    d_k = Q.shape[-1]

    scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(d_k)

    if mask is not None:
        if mask.ndim == 2:
            mask = mask[:, np.newaxis, :]
        scores = np.where(mask, scores, -1e9)
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attn_weights = exp_scores / (np.sum(exp_scores, axis=-1, keepdims=True) + 1e-9)

    #effectively weighted sum
    output = np.matmul(attn_weights, V)
    return output, attn_weights

batch_size, seq_len, d_k = 2, 4, 8

Q = np.random.randn(batch_size, seq_len, d_k).astype(np.float32)
K = np.random.randn(batch_size, seq_len, d_k).astype(np.float32)
V = np.random.randn(batch_size, seq_len, d_k).astype(np.float32)

#mask the last token
mask = np.array([[True, True, True, True],
                 [True, True, True, False]])

output, weights = scaled_dot_product_attention(Q, K, V, mask)

print("Input Q shape:", Q.shape)
print("Scores shape before mask:", np.matmul(Q, K.transpose(0, 2, 1)).shape)
print("Attention weights shape:", weights.shape)
print("Output shape:", output.shape)
print(weights[1, 0])

Input Q shape: (2, 4, 8)
Scores shape before mask: (2, 4, 4)
Attention weights shape: (2, 4, 4)
Output shape: (2, 4, 8)
[0.24695523 0.66921543 0.08382934 0.        ]


In [ ]:
#simple vocabulary class that contructs a dictionary based on the words in the sentences
class Vocab:
    def __init__(self):
        self.word2idx = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
        self.idx2word = {v: k for k, v in self.word2idx.items()}

    def build(self, sentences):
        counter = Counter()
        for sent in sentences:
            counter.update(sent.split())
        for word in counter:
            if word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

#a wrapper for the sentences that adds the sos and eos tokens
#implements the needed torch dataset functions
class TranslationDataset(Dataset):
    def __init__(self, src_sents, tgt_sents, src_vocab, tgt_vocab, max_len=30):
        self.src = []
        self.tgt = []
        for s, t in zip(src_sents, tgt_sents):
            src_tensor = self._sentence_to_tensor(s, src_vocab, max_len)
            tgt_tensor = self._sentence_to_tensor(t, tgt_vocab, max_len)
            self.src.append(src_tensor)
            self.tgt.append(tgt_tensor)

    def _sentence_to_tensor(self, sent, vocab, max_len):
        tokens = sent.split()
        indices = [vocab.word2idx.get(t, vocab.word2idx['<unk>']) for t in tokens]
        indices = [vocab.word2idx['<sos>']] + indices + [vocab.word2idx['<eos>']]
        if len(indices) > max_len:
            indices = indices[:max_len]
        else:
            indices += [vocab.word2idx['<pad>']] * (max_len - len(indices))
        return torch.tensor(indices, dtype=torch.long)

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx]

eng_vocab = Vocab()
rus_vocab = Vocab()
eng_vocab.build(eng_sentences)
rus_vocab.build(rus_sentences)

print(f"English vocab size: {len(eng_vocab.word2idx)}")
print(f"Russian vocab size: {len(rus_vocab.word2idx)}")

# 80/10/10 split
indices = list(range(len(eng_sentences)))
random.shuffle(indices)
train_idx = indices[:8000]
val_idx   = indices[8000:9000]
test_idx  = indices[9000:]

#create the torch datasets
train_dataset = TranslationDataset(
    [eng_sentences[i] for i in train_idx],
    [rus_sentences[i] for i in train_idx],
    eng_vocab, rus_vocab
)
val_dataset = TranslationDataset(
    [eng_sentences[i] for i in val_idx],
    [rus_sentences[i] for i in val_idx],
    eng_vocab, rus_vocab
)
test_dataset = TranslationDataset(
    [eng_sentences[i] for i in test_idx],
    [rus_sentences[i] for i in test_idx],
    eng_vocab, rus_vocab
)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

English vocab size: 1710
Russian vocab size: 4651


In [ ]:
#torch version of scaled dot product attention so it can be used in the model
#this is otherwise the same as the numpy version
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, Q, K, V, mask=None):
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = torch.softmax(scores, dim=-1)
        output = torch.matmul(attn, V)
        return output, attn

#encodes src text -> embedding -> rnn -> hidden
class Seq2SeqEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim, n_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        return outputs, hidden

#decodes hidden -> attention -> rnn -> logits out
class Seq2SeqDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers, attention, dropout=0.1):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim + hidden_dim, hidden_dim, n_layers, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(emb_dim + hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, hidden, encoder_outputs, mask=None):
        embedded = self.dropout(self.embedding(tgt))
        query = hidden[-1].unsqueeze(1)
        context, attn_weights = self.attention(query, encoder_outputs, encoder_outputs, mask)
        rnn_input = torch.cat((embedded, context), dim=-1)
        output, hidden = self.rnn(rnn_input, hidden)
        output = torch.cat((embedded, context, output), dim=-1)
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, attn_weights

#combines encoder and decoder with softmax
class Seq2SeqModel(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        tgt_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)
        #get encoding
        encoder_outputs, hidden = self.encoder(src)
        input = tgt[:, 0].unsqueeze(1)

        for t in range(1, tgt_len):
            output, hidden, _ = self.decoder(input, hidden, encoder_outputs)
            outputs[:, t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = tgt[:, t].unsqueeze(1) if teacher_force else top1.unsqueeze(1)
        return outputs

EMB_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 2
DROPOUT = 0.1

attn = ScaledDotProductAttention()
encoder = Seq2SeqEncoder(len(eng_vocab.word2idx), EMB_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
decoder = Seq2SeqDecoder(len(rus_vocab.word2idx), EMB_DIM, HIDDEN_DIM, N_LAYERS, attn, DROPOUT)
seq2seq_model = Seq2SeqModel(encoder, decoder, device).to(device)

In [ ]:
optimizer = optim.Adam(seq2seq_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=eng_vocab.word2idx['<pad>'])

def train_epoch(model, loader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = model(src, tgt)
        output = output[:, 1:].reshape(-1, output.size(-1))
        tgt = tgt[:, 1:].reshape(-1)
        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

def evaluate(model, loader, tgt_vocab):
    model.eval()
    references = []
    hypotheses = []
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt, teacher_forcing_ratio=0.0)
            output = output.argmax(-1)
            for ref, hyp in zip(tgt.cpu().numpy(), output.cpu().numpy()):
                ref_sent = [tgt_vocab.idx2word[i] for i in ref if i not in (0,1,2)]
                hyp_sent = [tgt_vocab.idx2word[i] for i in hyp if i not in (0,1,2)]
                references.append([ref_sent])
                hypotheses.append(hyp_sent)
    smoothie = SmoothingFunction().method4
    bleu = sum(sentence_bleu(ref, hyp, smoothing_function=smoothie) for ref, hyp in zip(references, hypotheses)) / len(references)
    return bleu

N_EPOCHS = 20
for epoch in range(N_EPOCHS):
    start = time.time()
    train_loss = train_epoch(seq2seq_model, train_loader, optimizer, criterion)
    bleu_val = evaluate(seq2seq_model, val_loader, rus_vocab)
    print(f"Epoch {epoch+1:2d} | Loss: {train_loss:.4f} | Val BLEU: {bleu_val:.4f} | Time: {time.time()-start:.1f}s")

test_bleu_seq2seq = evaluate(seq2seq_model, test_loader, rus_vocab)
print("Seq2Seq Test BLEU:", test_bleu_seq2seq)

Epoch  1 | Loss: 4.0972 | Val BLEU: 0.0666 | Time: 11.5s
Epoch  2 | Loss: 2.9289 | Val BLEU: 0.0943 | Time: 11.5s
Epoch  3 | Loss: 2.2637 | Val BLEU: 0.1165 | Time: 11.5s
Epoch  4 | Loss: 1.7266 | Val BLEU: 0.1384 | Time: 11.6s
Epoch  5 | Loss: 1.3059 | Val BLEU: 0.1148 | Time: 11.5s
Epoch  6 | Loss: 1.0800 | Val BLEU: 0.1559 | Time: 11.6s
Epoch  7 | Loss: 0.9856 | Val BLEU: 0.1605 | Time: 11.5s
Epoch  8 | Loss: 0.8882 | Val BLEU: 0.1598 | Time: 11.7s
Epoch  9 | Loss: 0.8389 | Val BLEU: 0.1651 | Time: 10.9s
Epoch 10 | Loss: 0.8004 | Val BLEU: 0.1566 | Time: 10.9s
Epoch 11 | Loss: 0.7753 | Val BLEU: 0.1572 | Time: 10.9s
Epoch 12 | Loss: 0.7372 | Val BLEU: 0.1523 | Time: 10.9s
Epoch 13 | Loss: 0.7368 | Val BLEU: 0.1380 | Time: 10.9s
Epoch 14 | Loss: 0.7258 | Val BLEU: 0.1526 | Time: 10.8s
Epoch 15 | Loss: 0.6980 | Val BLEU: 0.1519 | Time: 10.9s
Epoch 16 | Loss: 0.6833 | Val BLEU: 0.1537 | Time: 10.8s
Epoch 17 | Loss: 0.6720 | Val BLEU: 0.1592 | Time: 10.8s
Epoch 18 | Loss: 0.6918 | Val B

In [ ]:
#sinusoidal position encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

#multi head allowing for more complex text understanding
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention()

    def forward(self, Q, K, V, mask=None):
        batch = Q.size(0)

        Q = self.W_q(Q).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)

        context, _ = self.attention(Q, K, V, mask)
        context = context.transpose(1, 2).contiguous().view(batch, -1, self.d_model)
        return self.W_o(context)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=128, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=128, dropout=0.1):
        super().__init__()
        #self attention and cross attention
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        self_attn_out = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_out))
        cross_out = self.cross_attn(x, enc_out, enc_out, src_mask)
        x = self.norm2(x + self.dropout(cross_out))
        ff_out = self.ff(x)
        x = self.norm3(x + self.dropout(ff_out))
        return x

class SimplifiedTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=64, n_heads=2,
                 n_layers=2, d_ff=128, dropout=0.1, max_len=100):
        super().__init__()
        self.d_model = d_model
        self.src_embed = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.encoder = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        #create padding
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_len = tgt.size(1)
        causal = torch.tril(torch.ones((tgt_len, tgt_len), dtype=torch.bool, device=device))
        pad_mask = (tgt != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = causal.unsqueeze(0).unsqueeze(1) & pad_mask

        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)

        src_embed = self.dropout(self.pos_enc(self.src_embed(src) * math.sqrt(self.d_model)))
        tgt_embed = self.dropout(self.pos_enc(self.tgt_embed(tgt) * math.sqrt(self.d_model)))

        enc_out = src_embed
        for layer in self.encoder:
            enc_out = layer(enc_out, src_mask)

        dec_out = tgt_embed
        for layer in self.decoder:
            dec_out = layer(dec_out, enc_out, src_mask, tgt_mask)

        return self.fc_out(dec_out)

transformer = SimplifiedTransformer(
    len(eng_vocab.word2idx),
    len(rus_vocab.word2idx),
    d_model=64,
    n_heads=2,
    n_layers=2,
    d_ff=128,
    dropout=0.1
).to(device)

In [ ]:
optimizer_tf = optim.Adam(transformer.parameters(), lr=0.0005, eps=1e-9)
criterion_tf = nn.CrossEntropyLoss(ignore_index=eng_vocab.word2idx['<pad>'])

def train_transformer_epoch(model, loader, optimizer, criterion):
    model.train()
    epoch_loss = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = model(src, tgt[:, :-1])
        output = output.reshape(-1, output.size(-1))
        tgt_flat = tgt[:, 1:].reshape(-1)
        loss = criterion(output, tgt_flat)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

N_EPOCHS_TF = 20
for epoch in range(N_EPOCHS_TF):
    start = time.time()
    loss = train_transformer_epoch(transformer, train_loader, optimizer_tf, criterion_tf)
    print(f"Epoch {epoch+1:2d} | Loss: {loss:.4f} | Time: {time.time()-start:.1f}s")

def evaluate_transformer(model, loader, tgt_vocab, max_len=30):
    model.eval()
    references = []
    hypotheses = []
    with torch.no_grad():
        for src, tgt in loader:
            src = src.to(device)
            batch_size = src.size(0)

            #add sos token to the beginning
            tgt_input = torch.full((batch_size, 1), tgt_vocab.word2idx['<sos>'],
                                 dtype=torch.long, device=device)

            for t in range(1, max_len):
                #use greedy evaluation
                output = model(src, tgt_input)
                next_token = output[:, -1, :].argmax(dim=-1)

                tgt_input = torch.cat((tgt_input, next_token.unsqueeze(1)), dim=1)
                if (next_token == tgt_vocab.word2idx['<eos>']).all():
                    break

            for ref, hyp in zip(tgt.cpu().numpy(), tgt_input.cpu().numpy()):
                ref_sent = [tgt_vocab.idx2word[i] for i in ref
                            if i not in (tgt_vocab.word2idx['<pad>'],
                                        tgt_vocab.word2idx['<sos>'],
                                        tgt_vocab.word2idx['<eos>'])]
                hyp_sent = [tgt_vocab.idx2word[i] for i in hyp
                            if i not in (tgt_vocab.word2idx['<pad>'],
                                        tgt_vocab.word2idx['<sos>'],
                                        tgt_vocab.word2idx['<eos>'])]
                references.append([ref_sent])
                hypotheses.append(hyp_sent)

    smoothie = SmoothingFunction().method4
    bleu = sum(sentence_bleu(ref, hyp, smoothing_function=smoothie)
               for ref, hyp in zip(references, hypotheses)) / len(references)
    return bleu

bleu_val_tf = evaluate_transformer(transformer, val_loader, rus_vocab)
test_bleu_tf = evaluate_transformer(transformer, test_loader, rus_vocab)

print(f"Transformer Validation BLEU: {bleu_val_tf:.4f}")
print(f"Transformer Test BLEU : {test_bleu_tf:.4f}")

Epoch  1 | Loss: 5.3863 | Time: 2.1s
Epoch  2 | Loss: 3.7016 | Time: 2.1s
Epoch  3 | Loss: 3.3155 | Time: 2.1s
Epoch  4 | Loss: 3.0705 | Time: 2.1s
Epoch  5 | Loss: 2.8851 | Time: 2.1s
Epoch  6 | Loss: 2.7360 | Time: 2.1s
Epoch  7 | Loss: 2.5964 | Time: 2.1s
Epoch  8 | Loss: 2.4764 | Time: 2.1s
Epoch  9 | Loss: 2.3598 | Time: 2.1s
Epoch 10 | Loss: 2.2511 | Time: 2.1s
Epoch 11 | Loss: 2.1420 | Time: 2.1s
Epoch 12 | Loss: 2.0453 | Time: 2.1s
Epoch 13 | Loss: 1.9429 | Time: 2.1s
Epoch 14 | Loss: 1.8524 | Time: 2.1s
Epoch 15 | Loss: 1.7590 | Time: 2.1s
Epoch 16 | Loss: 1.6636 | Time: 2.0s
Epoch 17 | Loss: 1.5838 | Time: 2.1s
Epoch 18 | Loss: 1.5008 | Time: 2.1s
Epoch 19 | Loss: 1.4180 | Time: 2.0s
Epoch 20 | Loss: 1.3439 | Time: 2.1s
Transformer Validation BLEU: 0.0282
Transformer Test BLEU : 0.0283


In [ ]:
def translate_greedy_seq2seq(model, src_tensor, tgt_vocab, max_len=30):
    model.eval()
    with torch.no_grad():
        src = src_tensor.unsqueeze(0).to(device)
        tgt_input = torch.tensor([[tgt_vocab.word2idx['<sos>']]],
                                 dtype=torch.long, device=device)

        for _ in range(max_len - 1):
            output = model(src, tgt_input, teacher_forcing_ratio=0.0)
            next_token = output[:, -1, :].argmax(dim=-1)
            tgt_input = torch.cat((tgt_input, next_token.unsqueeze(1)), dim=1)

            if next_token.item() == tgt_vocab.word2idx['<eos>']:
                break
    return tgt_input.squeeze(0).cpu().numpy()


def translate_greedy_transformer(model, src_tensor, tgt_vocab, max_len=30):
    model.eval()
    with torch.no_grad():
        src = src_tensor.unsqueeze(0).to(device)
        tgt_input = torch.tensor([[tgt_vocab.word2idx['<sos>']]],
                                 dtype=torch.long, device=device)

        for _ in range(max_len - 1):
            output = model(src, tgt_input)
            next_token = output[:, -1, :].argmax(dim=-1)
            tgt_input = torch.cat((tgt_input, next_token.unsqueeze(1)), dim=1)

            if next_token.item() == tgt_vocab.word2idx['<eos>']:
                break
    return tgt_input.squeeze(0).cpu().numpy()


def print_translation_examples(num_examples=8):
    #get random examples from the test set
    example_indices = random.sample(range(len(test_dataset)), num_examples)

    for i, dataset_idx in enumerate(example_indices):
        src_tensor, tgt_tensor = test_dataset[dataset_idx]

        eng_words = [eng_vocab.idx2word[idx] for idx in src_tensor.tolist()
                     if idx not in (eng_vocab.word2idx['<pad>'],
                                    eng_vocab.word2idx['<sos>'],
                                    eng_vocab.word2idx['<eos>'])]
        ref_words = [rus_vocab.idx2word[idx] for idx in tgt_tensor.tolist()
                     if idx not in (rus_vocab.word2idx['<pad>'],
                                    rus_vocab.word2idx['<sos>'],
                                    rus_vocab.word2idx['<eos>'])]

        hyp_seq_idx = translate_greedy_seq2seq(seq2seq_model, src_tensor, rus_vocab)
        hyp_tf_idx   = translate_greedy_transformer(transformer, src_tensor, rus_vocab)

        hyp_seq_words = [rus_vocab.idx2word[idx] for idx in hyp_seq_idx
                         if idx not in (rus_vocab.word2idx['<pad>'],
                                        rus_vocab.word2idx['<sos>'],
                                        rus_vocab.word2idx['<eos>'])]
        hyp_tf_words  = [rus_vocab.idx2word[idx] for idx in hyp_tf_idx
                         if idx not in (rus_vocab.word2idx['<pad>'],
                                        rus_vocab.word2idx['<sos>'],
                                        rus_vocab.word2idx['<eos>'])]

        print(f"Example {i+1}")
        print(f"English (source) : {' '.join(eng_words)}")
        print(f"Reference Russian: {' '.join(ref_words)}")
        print(f"Seq2Seq prediction: {' '.join(hyp_seq_words)}")
        print(f"Transformer prediction: {' '.join(hyp_tf_words)}")
        print("-" * 90)
        print()

print_translation_examples(num_examples=8)

Example 1
English (source) : you tried .
Reference Russian: вы попробовали .
Seq2Seq prediction: ты попробовал .
Transformer prediction: ты попытался .
------------------------------------------------------------------------------------------

Example 2
English (source) : take it easy .
Reference Russian: не бери в голову .
Seq2Seq prediction: не парьтесь .
Transformer prediction: возьми это .
------------------------------------------------------------------------------------------

Example 3
English (source) : that hurts .
Reference Russian: больно .
Seq2Seq prediction: это разумно .
Transformer prediction: лови это .
------------------------------------------------------------------------------------------

Example 4
English (source) : i m not shy .
Reference Russian: я не стеснительная .
Seq2Seq prediction: я не стеснительный .
Transformer prediction: я не застенчивый .
------------------------------------------------------------------------------------------

Example 5
English (so

The transformer got a BLEU score of 0.0282 and the seq2seq model got a score of .141 on the test data. I don't see why that should be the case as just in reviewing the examples the translation attempts are modest by both models. They both can sometimes produce grammatically correct sentences and sometimes use correct verbs but other times they use vaguely related words or a combination of words that don't grammatically make sense. These sentences are short but if they were longer we might see that the transformer performs better because with multihead attention it would be better at capturing long term or complex sentence patterns. Especially in Russian there can be long term word dependencies where in a sentence qualifiers like "now" can move from the end of an english sentence to the beginning. I found that the epoch time was faster for the transformer (10s->2s). This is because input tokens can be batch processed by the transformer meanwhile the seq2seq model has to process token by token.